# GRC Security Intelligence - Phase 0: Reference setup

Creates schemas `grc_bronze`, `grc_silver`, `grc_gold`, `grc_ref`, the landing volume `grc_bronze.landing`, and the reference tables in `grc_ref`:

* `frameworks` (NIST CSF 2.0, ISO 27001:2022, SOC 2, CIS v8) and `framework_requirements`
* `controls` - unified control set (24 controls) and `control_crosswalk` (control -> framework requirement)
* `control_test_catalog` - one automated test per control
* `sla_policy`, `asset_tiers`, `control_effectiveness_bands`
* `business_units`, `risk_taxonomy`, `risk_scoring_scale`, `risk_rating_bands`

Idempotent: tables are overwritten on each run.

In [0]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("as_of_date", "2026-10-27")
CAT = dbutils.widgets.get("catalog")
AS_OF = dbutils.widgets.get("as_of_date")
REF = f"`{CAT}`.grc_ref"
for s, c in [("grc_bronze","GRC raw ingested security/GRC sources"),("grc_silver","GRC cleansed conformed entities"),
             ("grc_gold","GRC business-ready control, risk and compliance tables"),("grc_ref","GRC reference: frameworks, controls, crosswalk, SLAs")]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CAT}`.{s} COMMENT '{c}'")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CAT}`.grc_bronze.landing COMMENT 'Landing zone for simulated cloud storage and SharePoint drops'")

def save(name, rows, schema, comment):
    df = spark.createDataFrame(rows, schema)
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{REF}.{name}")
    spark.sql(f"COMMENT ON TABLE {REF}.{name} IS '{comment}'")
    return df.count()

In [0]:
frameworks = [
 ("NIST_CSF_2", "NIST Cybersecurity Framework", "2.0", "NIST", "2024-02-26", "Govern/Identify/Protect/Detect/Respond/Recover outcomes"),
 ("ISO_27001", "ISO/IEC 27001 Annex A", "2022", "ISO/IEC", "2022-10-25", "93 information security controls in 4 themes"),
 ("SOC2", "SOC 2 Trust Services Criteria", "2017 (rev. 2022)", "AICPA", "2022-10-01", "Common Criteria (Security) plus Availability"),
 ("CIS_V8", "CIS Critical Security Controls", "8.0", "Center for Internet Security", "2021-05-18", "18 controls / safeguards with IG1-IG3"),
]
save("frameworks", frameworks, "framework_id string, framework_name string, version string, publisher string, published_date string, description string",
     "Control frameworks in scope for GRC posture")

req = [
 ("NIST_CSF_2","GV.PO-01","Govern","Policy for managing cybersecurity risks is established and communicated"),
 ("NIST_CSF_2","GV.PO-02","Govern","Policy is reviewed, updated, communicated and enforced"),
 ("NIST_CSF_2","GV.RM-06","Govern","A standardized method for calculating, documenting, categorizing and prioritizing risks is established"),
 ("NIST_CSF_2","GV.SC-07","Govern","Supplier risks are identified, prioritized, assessed, responded to and monitored"),
 ("NIST_CSF_2","ID.AM-01","Identify","Inventories of hardware managed by the organization are maintained"),
 ("NIST_CSF_2","ID.RA-01","Identify","Vulnerabilities in assets are identified, validated and recorded"),
 ("NIST_CSF_2","ID.RA-05","Identify","Threats, vulnerabilities, likelihoods and impacts are used to understand inherent risk"),
 ("NIST_CSF_2","ID.RA-06","Identify","Risk responses are chosen, prioritized, planned, tracked and communicated"),
 ("NIST_CSF_2","PR.AA-01","Protect","Identities and credentials for authorized users, services and hardware are managed"),
 ("NIST_CSF_2","PR.AA-03","Protect","Users, services and hardware are authenticated"),
 ("NIST_CSF_2","PR.AA-05","Protect","Access permissions, entitlements and authorizations are managed, incorporating least privilege and separation of duties"),
 ("NIST_CSF_2","PR.AT-01","Protect","Personnel are provided with awareness and training"),
 ("NIST_CSF_2","PR.DS-01","Protect","Confidentiality, integrity and availability of data-at-rest are protected"),
 ("NIST_CSF_2","PR.DS-11","Protect","Backups of data are created, protected, maintained and tested"),
 ("NIST_CSF_2","PR.PS-01","Protect","Configuration management practices are established and applied"),
 ("NIST_CSF_2","PR.PS-02","Protect","Software is maintained, replaced and removed commensurate with risk"),
 ("NIST_CSF_2","PR.PS-04","Protect","Log records are generated and made available for continuous monitoring"),
 ("NIST_CSF_2","DE.CM-01","Detect","Networks and network services are monitored to find potentially adverse events"),
 ("NIST_CSF_2","DE.CM-09","Detect","Computing hardware and software, runtime environments and their data are monitored"),
 ("NIST_CSF_2","DE.AE-02","Detect","Potentially adverse events are analyzed to better understand associated activities"),
 ("NIST_CSF_2","RS.MA-01","Respond","The incident response plan is executed once an incident is declared"),
 ("NIST_CSF_2","RS.MI-01","Respond","Incidents are contained"),
 ("ISO_27001","A.5.1","Organizational","Policies for information security"),
 ("ISO_27001","A.5.3","Organizational","Segregation of duties"),
 ("ISO_27001","A.5.9","Organizational","Inventory of information and other associated assets"),
 ("ISO_27001","A.5.15","Organizational","Access control"),
 ("ISO_27001","A.5.16","Organizational","Identity management"),
 ("ISO_27001","A.5.17","Organizational","Authentication information"),
 ("ISO_27001","A.5.18","Organizational","Access rights"),
 ("ISO_27001","A.5.19","Organizational","Information security in supplier relationships"),
 ("ISO_27001","A.5.22","Organizational","Monitoring, review and change management of supplier services"),
 ("ISO_27001","A.5.26","Organizational","Response to information security incidents"),
 ("ISO_27001","A.6.3","People","Information security awareness, education and training"),
 ("ISO_27001","A.6.5","People","Responsibilities after termination or change of employment"),
 ("ISO_27001","A.8.2","Technological","Privileged access rights"),
 ("ISO_27001","A.8.5","Technological","Secure authentication"),
 ("ISO_27001","A.8.7","Technological","Protection against malware"),
 ("ISO_27001","A.8.8","Technological","Management of technical vulnerabilities"),
 ("ISO_27001","A.8.9","Technological","Configuration management"),
 ("ISO_27001","A.8.13","Technological","Information backup"),
 ("ISO_27001","A.8.15","Technological","Logging"),
 ("ISO_27001","A.8.16","Technological","Monitoring activities"),
 ("ISO_27001","A.8.24","Technological","Use of cryptography"),
 ("ISO_27001","A.8.32","Technological","Change management"),
 ("ISO_27001","Cl.6.1.2","Clause","Information security risk assessment"),
 ("SOC2","CC1.4","Control Environment","Commitment to attract, develop and retain competent individuals"),
 ("SOC2","CC2.2","Communication and Information","Internally communicates information, including objectives and responsibilities for internal control"),
 ("SOC2","CC3.2","Risk Assessment","Identifies risks to the achievement of objectives and analyzes risks"),
 ("SOC2","CC5.3","Control Activities","Deploys control activities through policies and procedures"),
 ("SOC2","CC6.1","Logical and Physical Access","Logical access security software, infrastructure and architectures"),
 ("SOC2","CC6.2","Logical and Physical Access","Registers, authorizes and removes system credentials"),
 ("SOC2","CC6.3","Logical and Physical Access","Authorizes, modifies or removes access based on roles, least privilege and segregation of duties"),
 ("SOC2","CC6.8","Logical and Physical Access","Prevents or detects unauthorized or malicious software"),
 ("SOC2","CC7.1","System Operations","Detection and monitoring procedures for configuration changes and new vulnerabilities"),
 ("SOC2","CC7.2","System Operations","Monitors system components for anomalies indicative of malicious acts"),
 ("SOC2","CC7.3","System Operations","Evaluates security events to determine whether they are incidents"),
 ("SOC2","CC7.4","System Operations","Responds to identified security incidents"),
 ("SOC2","CC8.1","Change Management","Authorizes, designs, tests, approves and implements changes"),
 ("SOC2","CC9.2","Risk Mitigation","Assesses and manages risks associated with vendors and business partners"),
 ("SOC2","A1.2","Availability","Environmental protections, software, data backup and recovery infrastructure"),
 ("CIS_V8","1.1","Inventory and Control of Enterprise Assets","Establish and maintain detailed enterprise asset inventory"),
 ("CIS_V8","3.11","Data Protection","Encrypt sensitive data at rest"),
 ("CIS_V8","4.1","Secure Configuration","Establish and maintain a secure configuration process"),
 ("CIS_V8","5.3","Account Management","Disable dormant accounts"),
 ("CIS_V8","5.4","Account Management","Restrict administrator privileges to dedicated administrator accounts"),
 ("CIS_V8","6.2","Access Control Management","Establish an access revoking process"),
 ("CIS_V8","6.3","Access Control Management","Require MFA for externally-exposed applications"),
 ("CIS_V8","6.5","Access Control Management","Require MFA for administrative access"),
 ("CIS_V8","6.8","Access Control Management","Define and maintain role-based access control"),
 ("CIS_V8","7.1","Continuous Vulnerability Management","Establish and maintain a vulnerability management process"),
 ("CIS_V8","7.5","Continuous Vulnerability Management","Perform automated vulnerability scans of internal enterprise assets"),
 ("CIS_V8","7.7","Continuous Vulnerability Management","Remediate detected vulnerabilities"),
 ("CIS_V8","8.2","Audit Log Management","Collect audit logs"),
 ("CIS_V8","8.11","Audit Log Management","Conduct audit log reviews"),
 ("CIS_V8","10.1","Malware Defenses","Deploy and maintain anti-malware software"),
 ("CIS_V8","11.2","Data Recovery","Perform automated backups"),
 ("CIS_V8","13.1","Network Monitoring and Defense","Centralize security event alerting"),
 ("CIS_V8","14.1","Security Awareness and Skills Training","Establish and maintain a security awareness program"),
 ("CIS_V8","15.4","Service Provider Management","Ensure service provider contracts include security requirements"),
 ("CIS_V8","15.5","Service Provider Management","Assess service providers"),
 ("CIS_V8","17.4","Incident Response Management","Establish and maintain an incident response process"),
 ("CIS_V8","17.8","Incident Response Management","Conduct post-incident reviews"),
]
save("framework_requirements", req, "framework_id string, requirement_id string, requirement_group string, requirement_title string",
     "Framework requirements (NIST CSF 2.0 subcategories, ISO 27001 Annex A, SOC 2 TSC, CIS v8 safeguards) referenced by the unified control set")

In [0]:
controls = [
 ("GRC-IAM-01","MFA enforced for workforce identities","Identity & Access","All active workforce identities are enrolled in and required to use MFA","IAM Lead","Preventive","Daily",True,True),
 ("GRC-IAM-02","MFA enforced for privileged accounts","Identity & Access","All privileged/admin accounts require phishing-resistant MFA","IAM Lead","Preventive","Daily",True,True),
 ("GRC-IAM-03","Timely deprovisioning of leavers","Identity & Access","Accounts of terminated personnel are disabled within 24 hours of HR termination","IAM Lead","Preventive","Daily",True,True),
 ("GRC-IAM-04","Dormant accounts disabled","Identity & Access","Accounts with no sign-in for 90+ days are disabled","IAM Lead","Corrective","Weekly",True,False),
 ("GRC-IAM-05","Privileged access reviewed","Identity & Access","Privileged entitlements are reviewed and recertified every 90 days","IAM Lead","Detective","Quarterly",True,True),
 ("GRC-IAM-06","Segregation of duties enforced","Identity & Access","No identity holds toxic entitlement combinations defined in the SoD ruleset","IAM Lead","Preventive","Daily",True,True),
 ("GRC-IAM-07","Root / break-glass usage monitored","Identity & Access","Cloud root and break-glass accounts are not used without MFA and an approved ticket","Cloud Security Lead","Detective","Daily",True,True),
 ("GRC-AST-01","Asset inventory complete","Asset Management","Every in-scope asset has an owner, business unit and criticality tier in the CMDB","IT Asset Manager","Detective","Daily",True,False),
 ("GRC-AST-02","EDR coverage","Asset Management","All servers and endpoints run a healthy EDR agent reporting in the last 7 days","Endpoint Security Lead","Preventive","Daily",True,True),
 ("GRC-VUL-01","Vulnerability scan coverage","Vulnerability Management","All in-scope assets are scanned at least every 7 days","Vulnerability Mgmt Lead","Detective","Weekly",True,False),
 ("GRC-VUL-02","Vulnerabilities remediated within SLA","Vulnerability Management","Open vulnerabilities are remediated within severity-based SLA","Vulnerability Mgmt Lead","Corrective","Daily",True,True),
 ("GRC-VUL-03","Known exploited vulnerabilities remediated","Vulnerability Management","CISA KEV-listed vulnerabilities are remediated within 14 days","Vulnerability Mgmt Lead","Corrective","Daily",True,True),
 ("GRC-CFG-01","No publicly exposed storage","Configuration Management","Cloud storage buckets/containers are not publicly accessible unless approved by exception","Cloud Security Lead","Preventive","Daily",True,True),
 ("GRC-CFG-02","Changes follow change management","Change Management","IAM and infrastructure changes in production have an approved change ticket","Change Manager","Detective","Daily",True,False),
 ("GRC-LOG-01","Audit logging enabled","Logging & Monitoring","Cloud audit logging (CloudTrail / Activity Log) is enabled in all accounts and regions","Cloud Security Lead","Detective","Daily",True,True),
 ("GRC-LOG-02","Security alerts triaged within SLA","Logging & Monitoring","EDR and SIEM alerts are triaged within severity-based SLA","SOC Manager","Detective","Daily",True,True),
 ("GRC-DAT-01","Encryption at rest","Data Protection","Storage and databases holding confidential data are encrypted at rest","Data Protection Lead","Preventive","Daily",True,True),
 ("GRC-DAT-02","Backups performed and tested","Data Protection","Tier 1 systems have successful daily backups and quarterly restore tests","Infrastructure Lead","Corrective","Daily",True,False),
 ("GRC-IR-01","Incidents contained within SLA","Incident Response","Declared security incidents are contained within severity-based SLA and have a post-incident review","SOC Manager","Corrective","Monthly",True,True),
 ("GRC-POL-01","Policies reviewed annually","Governance","Information security policies are reviewed and approved at least every 12 months","GRC Manager","Directive","Monthly",True,False),
 ("GRC-POL-02","Policy exceptions governed","Governance","Policy exceptions are approved, time-bound (max 180 days) and not expired","GRC Manager","Directive","Weekly",True,False),
 ("GRC-RSK-01","Risk register maintained","Risk Management","Risks have an owner, rating, treatment plan and review within 90 days","GRC Manager","Directive","Monthly",True,False),
 ("GRC-TRN-01","Security awareness training completed","People","At least 95% of workforce completed annual security awareness training","Security Awareness Lead","Preventive","Monthly",True,False),
 ("GRC-VND-01","Third-party risk assessed","Third-Party Risk","Critical and high vendors have a current (<12 months) security assessment","TPRM Lead","Detective","Monthly",True,False),
]
save("controls", controls,
     "control_id string, control_name string, control_domain string, control_description string, owner_role string, control_type string, test_frequency string, automated boolean, key_control boolean",
     "Unified internal control set (UCF) tested by the GRC platform and mapped to external frameworks")

xw = {
 "GRC-IAM-01": {"NIST_CSF_2":["PR.AA-03"],"ISO_27001":["A.8.5","A.5.17"],"SOC2":["CC6.1"],"CIS_V8":["6.3"]},
 "GRC-IAM-02": {"NIST_CSF_2":["PR.AA-03","PR.AA-05"],"ISO_27001":["A.8.2","A.8.5"],"SOC2":["CC6.1"],"CIS_V8":["6.5"]},
 "GRC-IAM-03": {"NIST_CSF_2":["PR.AA-01"],"ISO_27001":["A.6.5","A.5.18"],"SOC2":["CC6.2"],"CIS_V8":["6.2"]},
 "GRC-IAM-04": {"NIST_CSF_2":["PR.AA-01"],"ISO_27001":["A.5.16","A.5.18"],"SOC2":["CC6.2"],"CIS_V8":["5.3"]},
 "GRC-IAM-05": {"NIST_CSF_2":["PR.AA-05"],"ISO_27001":["A.8.2","A.5.18"],"SOC2":["CC6.3"],"CIS_V8":["5.4","6.8"]},
 "GRC-IAM-06": {"NIST_CSF_2":["PR.AA-05"],"ISO_27001":["A.5.3","A.5.15"],"SOC2":["CC6.3"],"CIS_V8":["6.8"]},
 "GRC-IAM-07": {"NIST_CSF_2":["PR.AA-05","DE.CM-09"],"ISO_27001":["A.8.2","A.8.16"],"SOC2":["CC6.1","CC7.2"],"CIS_V8":["5.4","8.11"]},
 "GRC-AST-01": {"NIST_CSF_2":["ID.AM-01"],"ISO_27001":["A.5.9"],"SOC2":["CC6.1"],"CIS_V8":["1.1"]},
 "GRC-AST-02": {"NIST_CSF_2":["DE.CM-09"],"ISO_27001":["A.8.7"],"SOC2":["CC6.8"],"CIS_V8":["10.1"]},
 "GRC-VUL-01": {"NIST_CSF_2":["ID.RA-01"],"ISO_27001":["A.8.8"],"SOC2":["CC7.1"],"CIS_V8":["7.5"]},
 "GRC-VUL-02": {"NIST_CSF_2":["PR.PS-02","ID.RA-01"],"ISO_27001":["A.8.8"],"SOC2":["CC7.1"],"CIS_V8":["7.1","7.7"]},
 "GRC-VUL-03": {"NIST_CSF_2":["PR.PS-02"],"ISO_27001":["A.8.8"],"SOC2":["CC7.1"],"CIS_V8":["7.7"]},
 "GRC-CFG-01": {"NIST_CSF_2":["PR.PS-01","PR.DS-01"],"ISO_27001":["A.8.9"],"SOC2":["CC6.1"],"CIS_V8":["4.1"]},
 "GRC-CFG-02": {"NIST_CSF_2":["PR.PS-01"],"ISO_27001":["A.8.32","A.8.9"],"SOC2":["CC8.1"],"CIS_V8":["4.1"]},
 "GRC-LOG-01": {"NIST_CSF_2":["PR.PS-04"],"ISO_27001":["A.8.15"],"SOC2":["CC7.2"],"CIS_V8":["8.2"]},
 "GRC-LOG-02": {"NIST_CSF_2":["DE.CM-01","DE.AE-02"],"ISO_27001":["A.8.16"],"SOC2":["CC7.2","CC7.3"],"CIS_V8":["13.1","8.11"]},
 "GRC-DAT-01": {"NIST_CSF_2":["PR.DS-01"],"ISO_27001":["A.8.24"],"SOC2":["CC6.1"],"CIS_V8":["3.11"]},
 "GRC-DAT-02": {"NIST_CSF_2":["PR.DS-11"],"ISO_27001":["A.8.13"],"SOC2":["A1.2"],"CIS_V8":["11.2"]},
 "GRC-IR-01":  {"NIST_CSF_2":["RS.MA-01","RS.MI-01"],"ISO_27001":["A.5.26"],"SOC2":["CC7.3","CC7.4"],"CIS_V8":["17.4","17.8"]},
 "GRC-POL-01": {"NIST_CSF_2":["GV.PO-01","GV.PO-02"],"ISO_27001":["A.5.1"],"SOC2":["CC5.3"],"CIS_V8":["14.1"]},
 "GRC-POL-02": {"NIST_CSF_2":["GV.PO-02","ID.RA-06"],"ISO_27001":["A.5.1"],"SOC2":["CC5.3","CC3.2"],"CIS_V8":["4.1"]},
 "GRC-RSK-01": {"NIST_CSF_2":["GV.RM-06","ID.RA-05"],"ISO_27001":["Cl.6.1.2"],"SOC2":["CC3.2"],"CIS_V8":["7.1"]},
 "GRC-TRN-01": {"NIST_CSF_2":["PR.AT-01"],"ISO_27001":["A.6.3"],"SOC2":["CC1.4","CC2.2"],"CIS_V8":["14.1"]},
 "GRC-VND-01": {"NIST_CSF_2":["GV.SC-07"],"ISO_27001":["A.5.19","A.5.22"],"SOC2":["CC9.2"],"CIS_V8":["15.4","15.5"]},
}
xw_rows = [(c, f, r, "Primary" if i == 0 else "Supporting") for c, m in xw.items() for f, rs in m.items() for i, r in enumerate(rs)]
save("control_crosswalk", xw_rows, "control_id string, framework_id string, requirement_id string, mapping_strength string",
     "Many-to-many mapping of unified controls to framework requirements")

In [0]:
tests = [
 ("T-IAM-01","GRC-IAM-01","Workforce MFA enrollment","Active workforce identities","mfa_enrolled = true","silver.identity","% identities passing","Table snapshot"),
 ("T-IAM-02","GRC-IAM-02","Privileged MFA enforcement","Active privileged identities","mfa_enrolled = true AND mfa_type IN (fido2, push_number_match)","silver.identity, silver.entitlement","% identities passing","Table snapshot"),
 ("T-IAM-03","GRC-IAM-03","Leaver deprovisioning timeliness","Identities with HR termination date","disabled_ts - termination_ts <= 24h","silver.identity","% leavers passing","Table snapshot"),
 ("T-IAM-04","GRC-IAM-04","Dormant account check","Enabled identities","last_signin within 90 days","silver.identity, bronze.idp_signins","% identities passing","Table snapshot"),
 ("T-IAM-05","GRC-IAM-05","Privileged access recertification","Privileged entitlements","last_review_date within 90 days","silver.entitlement","% entitlements passing","Access review export"),
 ("T-IAM-06","GRC-IAM-06","SoD conflict scan","Active identities","no toxic entitlement pair from SoD ruleset","silver.entitlement","% identities passing","Table snapshot"),
 ("T-IAM-07","GRC-IAM-07","Root account usage review","Root/break-glass sign-in events","mfa_used = true AND change ticket present","silver.security_event, silver.remediation_ticket","% events passing","CloudTrail events"),
 ("T-AST-01","GRC-AST-01","CMDB completeness","Active assets","owner, business_unit and tier populated","silver.asset","% assets passing","Table snapshot"),
 ("T-AST-02","GRC-AST-02","EDR agent health","Active servers and endpoints","edr_last_seen within 7 days","silver.asset","% assets passing","EDR inventory"),
 ("T-VUL-01","GRC-VUL-01","Scanner coverage","Active assets","last_scan_ts within 7 days","silver.asset, silver.vulnerability_finding","% assets passing","Scanner export"),
 ("T-VUL-02","GRC-VUL-02","Vulnerability SLA adherence","Open and recently closed findings","age_days <= sla_days for severity and asset tier","silver.vulnerability_finding, ref.sla_policy","% findings passing","Scanner export"),
 ("T-VUL-03","GRC-VUL-03","KEV remediation","Findings with kev_flag = true","age_days <= 14","silver.vulnerability_finding","% findings passing","Scanner export"),
 ("T-CFG-01","GRC-CFG-01","Public storage exposure","Cloud storage resources","public_access = false OR approved exception","silver.security_event, silver.risk_exception","% resources passing","CloudTrail events"),
 ("T-CFG-02","GRC-CFG-02","Unapproved change detection","Production IAM/infra change events","linked approved change ticket within window","silver.security_event, silver.remediation_ticket","% changes passing","CloudTrail events"),
 ("T-LOG-01","GRC-LOG-01","Audit logging enabled","Cloud accounts x regions","trail enabled and delivering in last 24h","silver.security_event","% accounts passing","CloudTrail config"),
 ("T-LOG-02","GRC-LOG-02","Alert triage timeliness","EDR/SIEM alerts","triage_ts - alert_ts <= SLA for severity","silver.security_event, ref.sla_policy","% alerts passing","EDR alerts"),
 ("T-DAT-01","GRC-DAT-01","Encryption at rest","Storage and DB assets with confidential data","encryption_at_rest = true","silver.asset","% assets passing","Table snapshot"),
 ("T-DAT-02","GRC-DAT-02","Backup success","Tier 1 assets","successful backup in last 24h","silver.asset","% assets passing","Backup report"),
 ("T-IR-01","GRC-IR-01","Incident containment SLA","Declared incidents","contain_ts - declare_ts <= SLA AND PIR completed","silver.remediation_ticket","% incidents passing","Incident tickets"),
 ("T-POL-01","GRC-POL-01","Policy review currency","Published policies","last_review_date within 365 days","silver.policy, silver.policy_document_extract","% policies passing","SharePoint policy library"),
 ("T-POL-02","GRC-POL-02","Exception validity","Active exceptions","approved AND expiry_date >= today AND duration <= 180 days","silver.risk_exception","% exceptions passing","SharePoint exceptions list"),
 ("T-RSK-01","GRC-RSK-01","Risk register hygiene","Open risks","owner present AND treatment plan present AND reviewed within 90 days","silver.risk","% risks passing","SharePoint risk register"),
 ("T-TRN-01","GRC-TRN-01","Training completion","Active workforce","completed annual training in last 365 days","silver.training_status","% workforce passing","LMS export"),
 ("T-VND-01","GRC-VND-01","Vendor assessment currency","Critical and high vendors","assessment_date within 365 days","silver.vendor","% vendors passing","Vendor questionnaires"),
]
save("control_test_catalog", tests,
     "test_id string, control_id string, test_name string, test_population string, pass_condition string, source_tables string, metric string, evidence_type string",
     "Automated control tests evaluated daily against silver tables")

In [0]:
sla = [
 ("VULN_REMEDIATION","Critical","days",15),("VULN_REMEDIATION","High","days",30),("VULN_REMEDIATION","Medium","days",90),("VULN_REMEDIATION","Low","days",180),
 ("KEV_REMEDIATION","Any","days",14),
 ("ALERT_TRIAGE","Critical","hours",1),("ALERT_TRIAGE","High","hours",4),("ALERT_TRIAGE","Medium","hours",24),("ALERT_TRIAGE","Low","hours",72),
 ("INCIDENT_CONTAINMENT","SEV1","hours",4),("INCIDENT_CONTAINMENT","SEV2","hours",24),("INCIDENT_CONTAINMENT","SEV3","hours",72),
 ("LEAVER_DEPROVISION","Any","hours",24),("DORMANT_ACCOUNT","Any","days",90),("PRIV_ACCESS_REVIEW","Any","days",90),
 ("POLICY_REVIEW","Any","days",365),("EXCEPTION_MAX_DURATION","Any","days",180),("RISK_REVIEW","Any","days",90),
 ("TRAINING_RECERT","Any","days",365),("VENDOR_REASSESSMENT","Critical","days",365),("VENDOR_REASSESSMENT","High","days",365),("VENDOR_REASSESSMENT","Medium","days",730),
 ("SCAN_FREQUENCY","Any","days",7),("EDR_HEARTBEAT","Any","days",7),
]
save("sla_policy", sla, "sla_type string, severity string, unit string, sla_value int", "Service level targets used by control tests")

tiers = [("T0","Crown jewel","Business-critical systems holding regulated or highly confidential data",0.5),
         ("T1","Critical","Production systems supporting key business processes",1.0),
         ("T2","Important","Internal production systems",1.0),
         ("T3","Standard","Non-production, labs and end-user devices",1.5)]
save("asset_tiers", tiers, "tier_code string, tier_name string, description string, sla_multiplier double",
     "Asset criticality tiers; sla_multiplier scales vulnerability SLA days")

bands = [("Effective",95.0,100.0,"#2E7D32",1),("Partially Effective",80.0,95.0,"#F9A825",2),("Ineffective",0.0,80.0,"#C62828",3)]
save("control_effectiveness_bands", bands, "status string, min_pass_pct double, max_pass_pct double, color_hex string, sort_order int",
     "Pass-rate bands: Effective >=95, Partially 80 to <95, Ineffective <80")

bus = [("BU-CORP","Corporate IT","CIO","EMEA","Low"),("BU-FIN","Finance","CFO","Global","Low"),
       ("BU-ENG","Engineering","CTO","Americas","Medium"),("BU-SAL","Sales & Marketing","CRO","Americas","Medium"),
       ("BU-OPS","Operations","COO","APAC","Medium")]
save("business_units", bus, "bu_id string, bu_name string, executive_owner string, region string, risk_appetite string", "Business units used for posture slicing")

tax = [("RT-IAM","Cyber","Identity & access compromise","Account takeover, privilege misuse, orphaned access"),
       ("RT-VUL","Cyber","Exploitation of vulnerabilities","Unpatched or misconfigured systems exploited"),
       ("RT-CLD","Cyber","Cloud misconfiguration","Public exposure or insecure cloud settings"),
       ("RT-DAT","Cyber","Data breach / leakage","Loss of confidentiality of regulated data"),
       ("RT-MAL","Cyber","Malware / ransomware","Malicious code impacting availability or integrity"),
       ("RT-INS","Cyber","Insider threat","Malicious or negligent insider activity"),
       ("RT-TPR","Third Party","Supplier security failure","Vendor breach or service failure"),
       ("RT-RES","Operational","Operational resilience","Loss of critical services, backup/recovery failure"),
       ("RT-CMP","Compliance","Regulatory non-compliance","Failure to meet regulatory or contractual obligations")]
save("risk_taxonomy", tax, "risk_category_id string, risk_domain string, risk_category string, description string", "Risk taxonomy for the risk register")

scale = [(1,"Rare","Negligible"),(2,"Unlikely","Minor"),(3,"Possible","Moderate"),(4,"Likely","Major"),(5,"Almost certain","Severe")]
save("risk_scoring_scale", scale, "score int, likelihood_label string, impact_label string", "1-5 likelihood and impact scales")

rating = [("Low",1,4,"#2E7D32"),("Medium",5,9,"#F9A825"),("High",10,15,"#EF6C00"),("Critical",16,25,"#C62828")]
save("risk_rating_bands", rating, "rating string, min_score int, max_score int, color_hex string", "Risk rating bands on likelihood x impact (1-25)")

In [0]:
%sql
USE CATALOG IDENTIFIER(:catalog);
SELECT 'orphan_xw_control' chk, count(*) n FROM grc_ref.control_crosswalk x LEFT ANTI JOIN grc_ref.controls c USING (control_id)
UNION ALL SELECT 'orphan_xw_requirement', count(*) FROM grc_ref.control_crosswalk x LEFT ANTI JOIN grc_ref.framework_requirements r USING (framework_id, requirement_id)
UNION ALL SELECT 'unused_requirements', count(*) FROM grc_ref.framework_requirements r LEFT ANTI JOIN grc_ref.control_crosswalk x USING (framework_id, requirement_id)
UNION ALL SELECT 'controls_not_in_4_fw', count(*) FROM (SELECT c.control_id FROM grc_ref.controls c LEFT JOIN grc_ref.control_crosswalk x USING (control_id) GROUP BY 1 HAVING count(DISTINCT x.framework_id) <> 4)
UNION ALL SELECT 'controls_without_test', count(*) FROM grc_ref.controls c LEFT ANTI JOIN grc_ref.control_test_catalog t USING (control_id)
UNION ALL SELECT 'rows_controls', count(*) FROM grc_ref.controls
UNION ALL SELECT 'rows_requirements', count(*) FROM grc_ref.framework_requirements
UNION ALL SELECT 'rows_crosswalk', count(*) FROM grc_ref.control_crosswalk